In [ ]:
# @title Setup (chạy ô này trước)
# Colab bắt đầu với một máy trống — clone repo và cài dependency.
import os, subprocess, sys

REPO = "https://github.com/hieutrungdao/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("Day21-Track3-Finetuning-Lab")
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > T4 GPU")


# NB3 — Huấn luyện cấu hình ĐÚNG

Cấu hình ở đây là "vùng không hối tiếc" của deck §10, viết thẳng thành code:

| Nút | Giá trị | Deck |
|---|---|---|
| `target_modules` | **toàn bộ linear của text decoder** | §10.2 |
| `learning_rate` | **≈10× LR full-FT** | §10.3 |
| batch hiệu dụng | **< 32** | §10.4 |
| `packing` | **tắt** — xem ghi chú | §13.3 |
| `padding_free` | chỉ khi có flash-attn **và** batch ≥ 2 | §13.3 |

> **Vì sao khác deck §15.** Deck khuyến nghị bật `packing` + `padding_free`. Trên
> model mặc định của lab, cả hai đều **không dùng được**, và lab nói thẳng thay vì
> bật cờ vô tác dụng:
>
> * `packing` **tắt** vì ta nạp nhãn đã token hoá sẵn (mask đã kiểm chứng ở NB1).
>   Packing nối các mẫu lại và sẽ phá vỡ căn chỉnh nhãn. *Tính đúng của mask quan
>   trọng hơn thông lượng.*
> * `padding_free` cần kernel **FlashAttention**, mà FA-2 đòi **Ampere (sm_80+)** —
>   T4 là Turing, không bao giờ có. Và với `batch=1` thì cũng chẳng có padding nào
>   để bỏ. Xem `labkit/device.py`.
>
> Tinh thần §13.3 vẫn đúng: *tăng tốc chỉ miễn phí khi ranh giới chuỗi được tôn
> trọng.* Ở đây điều kiện đó không thoả, nên ta không bật.
| `loss_type` | `chunked_nll` | §15 |
| `alpha` | `2r` | §9.3 |

In [ ]:
import json, os, pathlib, sys, time
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from labkit import data, device, generate, modeling, report, train
from labkit.config import SPECS, get_tier, training_epochs

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "data").exists() else pathlib.Path.cwd().parent
TIER = get_tier(os.environ.get("COMPUTE_TIER", "T4"))
SPEC = SPECS["correct"]
print(f"{TIER.name} · {TIER.model_id} · {SPEC.label}")
print(device.banner())      # which precision is ACTUALLY being used, and why

## 1. Nạp model — và nhìn vào kiến trúc bạn đang fine-tune

Deck §6.4 nói các base 2026 xen kẽ **linear attention** với **full attention**. Đây là
chỗ điều đó thôi là slide: config của chính model sẽ nói cho bạn biết.

In [ ]:
model, tok = generate.load_base(TIER, load_in_4bit=SPEC.load_in_4bit)
print(json.dumps(modeling.layer_type_summary(model.config), ensure_ascii=False, indent=2))

## 2. `all-linear` — nhưng không phải *mọi* linear

Qwen3.5 là model **đa phương thức**: text decoder + vision tower. `target_modules=
"all-linear"` của PEFT sẽ gắn adapter vào **cả vision encoder** bạn không hề huấn
luyện — adapter phình to, step chậm hơn, và merge ra một checkpoint sai.

`resolve_target_modules` trả về đúng phần text decoder.

In [ ]:
targets = modeling.resolve_target_modules(model, SPEC.target)
trainable = modeling.count_lora_params(model, targets, SPEC.r)
print(f"placement={SPEC.target}  modules={targets}")
print(f"trainable LoRA params ≈ {trainable/1e6:.2f} M")

for row in modeling.describe_placement(model, SPEC.r):
    print("   ", row)

## 3. Dataset đã tokenize + mask (dùng lại NB1)

In [ ]:
from datasets import Dataset

split_dir = ROOT / "data" / "split"
assert split_dir.exists(), "Chạy NB1 trước — chưa có data/split/"

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

train_rows = load_jsonl(split_dir / "train.jsonl")

# Small replay set to reduce catastrophic forgetting.
replay_rows = load_jsonl(ROOT / "data" / "train_replay.jsonl")

print(f"original train rows = {len(train_rows)}")
print(f"replay rows = {len(replay_rows)}")

train_rows = train_rows + replay_rows

print(f"total train rows = {len(train_rows)}")

MASK_MODE = os.environ.get("MASK_MODE", "assistant-only")

# Train on the mask you PROVED in NB1 — not on a library flag.
#
# TRL's `assistant_only_loss` builds its mask from `{% generation %}` markers in the
# chat template. Qwen3.5 has none, so that flag supervises ZERO tokens while emitting
# only a warning: training completes, the loss curve looks fine, the run is worthless.
# Check it yourself:  python scripts/check_mask_agreement.py
rows = data.to_training_dataset(tok, train_rows, max_length=TIER.max_length,
                                mask_mode=MASK_MODE)
train_ds = Dataset.from_list(rows)
sup = sum(sum(1 for x in r["labels"] if x != data.IGNORE_INDEX) for r in rows)
tot = sum(len(r["labels"]) for r in rows)
print(train_ds)
print(f"mask_mode = {MASK_MODE}   supervised {sup}/{tot} tokens ({sup/tot:.1%})")
assert 0 < sup < tot, "mask covers nothing or everything — stop and re-run NB1"

## 4. Cấu hình — lọc theo phiên bản TRL đang cài

`filter_kwargs` hỏi TRL xem nó nhận tham số nào và **bỏ những gì không nhận, có cảnh
báo**. Lab cũ phải monkey-patch `tokenizer=` → `processing_class=` và
`evaluation_strategy` → `eval_strategy`; những bản vá đó là hoá thạch của TRL trước
1.0. Cách này không tạo ra hoá thạch mới.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

EPOCHS = training_epochs()          # $EPOCHS, default 2 -- NB4 reads the SAME function
STEPS = train.planned_steps(len(rows), TIER, EPOCHS)
print(f"epochs={EPOCHS}  ->  {STEPS} optimizer steps  (NB4 runs its contrasts at exactly this many)")

want_sft = train.sft_config_kwargs(
    TIER, SPEC, output_dir=str(ROOT / "adapters" / SPEC.key),
    num_train_epochs=EPOCHS, mask_mode=MASK_MODE,
    total_steps=STEPS,
)
sft_kwargs, dropped = train.filter_kwargs(SFTConfig, want_sft, label="SFTConfig")
if dropped:
    print("⚠ TRL không nhận:", dropped)

want_lora = train.lora_config_kwargs(SPEC, targets)
lora_kwargs, _ = train.filter_kwargs(LoraConfig, want_lora, label="LoraConfig")

print(json.dumps({k: str(v) for k, v in sft_kwargs.items()}, indent=2)[:900])

## 5. Train

In [ ]:
generate.free_memory()
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(**sft_kwargs),
    train_dataset=train_ds,
    processing_class=tok,          # NOT tokenizer= — removed in TRL v1
    peft_config=LoraConfig(**lora_kwargs),
)

# TRL casts LoRA weights to bf16 regardless of the device or the fp16 flag it was
# handed. fp16's GradScaler cannot unscale bf16 gradients -- see F-23 and
# scripts/probe_precision.py. No-op on bf16/fp32 hardware.
fix = train.align_trainable_precision(trainer.model)
print("precision fix:", fix)

t0 = time.perf_counter()
result = trainer.train()
elapsed = time.perf_counter() - t0
print(f"train {elapsed:.0f}s  final loss {result.training_loss:.4f}")

## 6. LƯU ADAPTER NGAY

Trước khi eval. Eval có thể OOM; adapter thì đã an toàn trên đĩa.

In [ ]:
out = ROOT / "adapters" / SPEC.key
trainer.model.save_pretrained(out)
tok.save_pretrained(out)
print("saved ->", out)

row = train.summarize_run(SPEC, TIER, targets, trainable, elapsed, generate.peak_vram_gb())
row["final_loss"] = round(result.training_loss, 4)
row["mask_mode"] = MASK_MODE
# Record the step budget so NB5/verify can CHECK that the four runs are comparable,
# instead of trusting that they were configured the same way.
row["max_steps"] = STEPS
report.append_row(row, results_dir=ROOT / "results")
print(json.dumps(row, ensure_ascii=False, indent=2))

## ✅ Checkpoint NB3
- [ ] `adapters/correct/` tồn tại
- [ ] `results/runs.csv` có một dòng `correct`
- [ ] Bạn đã ghi lại loss cuối và peak VRAM